# Hospital Management – Data Engineering Mini Project

Dataset: Hospital Management Dataset (Kaggle) [link](https://www.kaggle.com/datasets/kanakbaghel/hospital-management-dataset)

Goal: Design and prepare a relational database structure for hospital operations

# Proposed Tables:
- patients (PK: patient_id)
- doctors (PK: doctor_id)
- appointments (FK: patient_id, doctor_id)
- treatments (FK: appointment_id)
- billing (FK: appointment_id)

# Visual: [link]

In [1]:
import pandas as pd

In [6]:
# Load raw patients dataset
patients_df = pd.read_csv("Raw Data/patients.csv")

In [3]:
# Create a working copy for cleaning
patients_clean = patients_df.copy()

In [3]:
#Checking basic info regarding patients table
patients_clean.info()
patients_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   patient_id          50 non-null     object
 1   first_name          50 non-null     object
 2   last_name           50 non-null     object
 3   gender              50 non-null     object
 4   date_of_birth       50 non-null     object
 5   contact_number      50 non-null     int64 
 6   address             50 non-null     object
 7   registration_date   50 non-null     object
 8   insurance_provider  50 non-null     object
 9   insurance_number    50 non-null     object
 10  email               50 non-null     object
dtypes: int64(1), object(10)
memory usage: 4.4+ KB


,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com


In [4]:
# Convert date_of_birth and registration_date to datetime format
patients_clean['date_of_birth'] = pd.to_datetime(patients_clean['date_of_birth'], errors='coerce')
patients_clean['registration_date'] = pd.to_datetime(patients_clean['registration_date'], errors='coerce')

# Convert contact_number to string to preserve formatting and leading digits
patients_clean['contact_number'] = patients_clean['contact_number'].astype(str)

# Normalize text fields (strip whitespace, apply capitalization)
patients_clean['first_name'] = patients_clean['first_name'].str.strip().str.title()
patients_clean['last_name'] = patients_clean['last_name'].str.strip().str.title()
patients_clean['address'] = patients_clean['address'].str.strip().str.title()
patients_clean['insurance_provider'] = patients_clean['insurance_provider'].str.strip().str.title()

# Normalize insurance_number format (uppercase, no spaces)
patients_clean['insurance_number'] = patients_clean['insurance_number'].astype(str).str.strip().str.upper()

# Normalize email formatting
patients_clean['email'] = patients_clean['email'].str.strip().str.lower()

# Check for duplicate patient_id values
assert not patients_clean['patient_id'].duplicated().any(), "Duplicate patient_id found!"

In [5]:
#Save Cleaned patients file to .csv
patients_clean.to_csv("sql_ready/patients_clean.csv", index=False)

In [5]:
# Load raw doctors dataset
doctors_df = pd.read_csv("Raw Data/doctors.csv")

In [8]:
# Create a working copy for cleaning
doctors_clean = doctors_df.copy()

In [8]:
#Checking basic info regarding patients table
doctors_clean.info()
doctors_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   doctor_id         10 non-null     object
 1   first_name        10 non-null     object
 2   last_name         10 non-null     object
 3   specialization    10 non-null     object
 4   phone_number      10 non-null     int64 
 5   years_experience  10 non-null     int64 
 6   hospital_branch   10 non-null     object
 7   email             10 non-null     object
dtypes: int64(2), object(6)
memory usage: 768.0+ bytes


,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email
0,D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com
2,D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com
3,D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com
4,D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com


In [9]:
# Convert phone_number to string to preserve leading zeros and formatting
doctors_clean['phone_number'] = doctors_clean['phone_number'].astype(str)

# Clean whitespace and apply consistent formatting
# Capitalize names and specializations
doctors_clean['first_name'] = doctors_clean['first_name'].str.strip().str.title()
doctors_clean['last_name'] = doctors_clean['last_name'].str.strip().str.title()
doctors_clean['specialization'] = doctors_clean['specialization'].str.strip().str.title()
doctors_clean['hospital_branch'] = doctors_clean['hospital_branch'].str.strip().str.title()

# Normalize email casing and trim spaces
doctors_clean['email'] = doctors_clean['email'].str.strip().str.lower()

# Check for duplicate doctor_id (should be unique for primary key)
assert not doctors_clean['doctor_id'].duplicated().any(), "Duplicate doctor_id found!"

# Validate years of experience (e.g., not negative or unrealistically high)
doctors_clean = doctors_clean[
    (doctors_clean['years_experience'] >= 0) & (doctors_clean['years_experience'] <= 60)
]

In [10]:
# Save the cleaned dataset to a CSV file
doctors_clean.to_csv("sql_ready/doctors_clean.csv", index=False)

In [2]:
# Load raw appointments dataset
appointments_df = pd.read_csv("Raw Data/appointments.csv")

In [3]:
# Create a working copy for cleaning
appointments_clean = appointments_df.copy()

In [3]:
#Checking basic info regarding appointments table
appointments_clean.info()
appointments_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   appointment_id    200 non-null    object
 1   patient_id        200 non-null    object
 2   doctor_id         200 non-null    object
 3   appointment_date  200 non-null    object
 4   appointment_time  200 non-null    object
 5   reason_for_visit  200 non-null    object
 6   status            200 non-null    object
dtypes: object(7)
memory usage: 11.1+ KB


,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled
1,A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show
2,A003,P048,D004,2023-06-28,8:00:00,Consultation,Cancelled
3,A004,P025,D006,2023-09-01,9:15:00,Consultation,Cancelled
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show


In [7]:
# Convert appointment_date to datetime
appointments_clean['appointment_date'] = pd.to_datetime(
    appointments_clean['appointment_date'], errors='coerce'
)

# Convert appointment_time to time object
appointments_clean['appointment_time'] = pd.to_datetime(
    appointments_clean['appointment_time'], format='%H:%M:%S', errors='coerce'
).dt.time

# Clean and standardize text fields
appointments_clean['reason_for_visit'] = appointments_clean['reason_for_visit'].str.strip().str.title()
appointments_clean['status'] = appointments_clean['status'].str.strip().str.title()

# Check for duplicate appointment IDs
assert not appointments_clean['appointment_id'].duplicated().any(), "Duplicate appointment_id found!"

# Validate foreign key references (if reference tables are loaded)
valid_patient_ids = patients_df['patient_id'].dropna().unique()
valid_doctor_ids = doctors_df['doctor_id'].dropna().unique()

# Keep only valid references to patients and doctors
appointments_clean = appointments_clean[
    appointments_clean['patient_id'].isin(valid_patient_ids) &
    appointments_clean['doctor_id'].isin(valid_doctor_ids)
]

In [15]:
# Save cleaned dataset to CSV
appointments_clean.to_csv("sql_ready/appointments_clean.csv", index=False)

In [10]:
# Load raw billing dataset
billing_df = pd.read_csv("Raw Data/billing.csv")

In [11]:
# Create a working copy for cleaning
billing_clean = billing_df.copy()

In [11]:
#Checking basic info regarding billing table
billing_clean.info()
billing_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bill_id         200 non-null    object 
 1   patient_id      200 non-null    object 
 2   treatment_id    200 non-null    object 
 3   bill_date       200 non-null    object 
 4   amount          200 non-null    float64
 5   payment_method  200 non-null    object 
 6   payment_status  200 non-null    object 
dtypes: float64(1), object(6)
memory usage: 11.1+ KB


,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status
0,B001,P034,T001,2023-08-09,3941.97,Insurance,Pending
1,B002,P032,T002,2023-06-09,4158.44,Insurance,Paid
2,B003,P048,T003,2023-06-28,3731.55,Insurance,Paid
3,B004,P025,T004,2023-09-01,4799.86,Insurance,Failed
4,B005,P040,T005,2023-07-06,582.05,Credit Card,Pending


In [13]:
# Convert bill_date to datetime format
billing_clean['bill_date'] = pd.to_datetime(billing_clean['bill_date'], errors='coerce')

# Standardize text fields (strip whitespace, title case or uppercase where relevant)
billing_clean['payment_method'] = billing_clean['payment_method'].str.strip().str.title()
billing_clean['payment_status'] = billing_clean['payment_status'].str.strip().str.title()

# Check for duplicate bill_id (should be unique)
assert not billing_clean['bill_id'].duplicated().any(), "Duplicate bill_id found!"

# Validate foreign keys if patients and treatments tables are loaded
valid_patient_ids = patients_df['patient_id'].dropna().unique()
billing_clean = billing_clean[billing_clean['patient_id'].isin(valid_patient_ids)]

In [16]:
# Save cleaned billing data to CSV
billing_clean.to_csv("sql_ready/billing_clean.csv", index=False)

In [17]:
# Load raw treatments dataset
treatments_df = pd.read_csv("Raw Data/treatments.csv")

In [18]:
# Create a working copy for cleaning
treatments_clean = treatments_df.copy()

In [19]:
#Checking basic info regarding treatments table
treatments_clean.info()
treatments_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   treatment_id    200 non-null    object 
 1   appointment_id  200 non-null    object 
 2   treatment_type  200 non-null    object 
 3   description     200 non-null    object 
 4   cost            200 non-null    float64
 5   treatment_date  200 non-null    object 
dtypes: float64(1), object(5)
memory usage: 9.5+ KB


,treatment_id,appointment_id,treatment_type,description,cost,treatment_date
0,T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09
1,T002,A002,MRI,Advanced protocol,4158.44,2023-06-09
2,T003,A003,MRI,Standard procedure,3731.55,2023-06-28
3,T004,A004,MRI,Basic screening,4799.86,2023-09-01
4,T005,A005,ECG,Standard procedure,582.05,2023-07-06


In [20]:
# Convert treatment_date to datetime format
treatments_clean['treatment_date'] = pd.to_datetime(
    treatments_clean['treatment_date'], errors='coerce'
)

# Standardize text fields (strip whitespace and title case)
treatments_clean['treatment_type'] = treatments_clean['treatment_type'].str.strip().str.title()
treatments_clean['description'] = treatments_clean['description'].str.strip().str.capitalize()

# Check for duplicate treatment_id
assert not treatments_clean['treatment_id'].duplicated().any(), "Duplicate treatment_id found!"

# Optional: validate appointment_id references if appointments table is loaded
valid_appointment_ids = appointments_clean['appointment_id'].dropna().unique()
treatments_clean = treatments_clean[
    treatments_clean['appointment_id'].isin(valid_appointment_ids)
]

In [21]:
# Save cleaned dataset to CSV
treatments_clean.to_csv("sql_ready/treatments_clean.csv", index=False)